# CatVTON × MAEVEN: virtual try-on trên Kaggle (tối ưu cho GPU T4 x2)

Bước 1 trong lộ trình học (`research/README.md`). Mục tiêu: hiểu trọn quy trình **tách người → mask vùng áo → diffusion**, và xem model xử lý vải lanh, bông dệt chéo của MAEVEN ra sao.

> **License:** code và weights của CatVTON là **CC BY-NC-SA 4.0**. Chỉ dùng để học và thử nghiệm. **Không** đưa ảnh tạo ra lên site bán hàng.

### Thiết lập trên Kaggle
- *Settings → Accelerator*: **GPU T4 x2**. *Internet*: **On**.
- *Add Input*: dataset có `garments/` và `persons/` (đặt tên gì cũng được).
- Mỗi khi đổi cài đặt ở mục 2, chạy lại **từ đầu**: *Run → Restart & clear cell outputs* rồi *Run All*.

### Notebook này tối ưu những gì cho T4
| Vấn đề trên T4 | Cách xử lý |
|---|---|
| T4 không có bf16 thật, PyTorch mới vẫn báo là có | Chọn **fp16** theo compute capability, không dùng `is_bf16_supported()` |
| Attention 24.576 điểm có thể đòi 36 GB | Ép kernel **memory-efficient**; không được thì **chia khúc** |
| Safety checker tốn thêm ~0,6 GB mỗi GPU | **Bỏ**: chỉ thử ảnh người mẫu mặc đồ bình thường |
| DensePose ghi file tạm chung một thư mục, không an toàn khi chạy song song | Tạo mask **tuần tự** và **cache** theo từng người, dùng lại cho mọi sản phẩm |
| Có 2 GPU | Mỗi GPU một pipeline, chạy **song song** hai ảnh một lúc |
| Máy bị treo vì CPU 100% | Giới hạn số luồng CPU của PyTorch |


## 1. Cài đặt
Phiên bản `diffusers` / `transformers` được ghim theo bản repo đã test.

In [ ]:
!git clone -q --depth 1 https://github.com/Zheng-Chong/CatVTON.git /kaggle/working/CatVTON 2>/dev/null || echo "Repo đã có sẵn"
!pip install -q "diffusers==0.36.0" "transformers==4.46.3" "accelerate>=0.31" "peft>=0.17" \
    "huggingface_hub>=0.34,<2.0" fvcore omegaconf pycocotools cloudpickle av

## 2. Cài đặt chạy

- **`FAST = True`**: 384×512, 20 bước. Nhanh, dùng để kiểm tra mọi thứ chạy được.
- **`FAST = False`**: 768×1024, đúng độ phân giải model được train. Dùng để chấm chất lượng.

Bắt đầu bằng `FAST = True`. Chạy ổn rồi mới chuyển sang `False`.

In [ ]:
FAST = True
W, H = (384, 512) if FAST else (768, 1024)
STEPS = 20 if FAST else 30
GUIDANCE = 2.5
SEED = 42

In [ ]:
import os
# Phải đặt trước khi import torch: giảm phân mảnh bộ nhớ GPU
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, gc, time, json, queue, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from types import SimpleNamespace

REPO = Path("/kaggle/working/CatVTON")
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_num_threads(2)  # Kaggle có 4 vCPU; để dành cho giao diện và đọc ảnh

assert torch.cuda.is_available(), "Bật GPU trong Settings → Accelerator"
GPUS = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
# bf16 chỉ có phần cứng từ Ampere (sm80). Không dùng is_bf16_supported(): bản PyTorch mới trả True cả trên T4.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
for g in GPUS:
    print(g, torch.cuda.get_device_name(g), f"{torch.cuda.get_device_properties(g).total_memory / 1e9:.1f} GB")
print("dtype:", DTYPE, "· độ phân giải:", f"{W}x{H}", "· steps:", STEPS)

## 3. Attention không tràn bộ nhớ

CatVTON ghép ảnh người và ảnh áo thành một ảnh cao gấp đôi. Ở 768×1024, chuỗi attention dài 24.576 điểm. Nếu PyTorch dùng cách tính "math", nó tạo cả ma trận 24.576 × 24.576 ở fp32 (~36 GB). Đoạn dưới ép dùng kernel memory-efficient, nếu không được thì chia query thành từng khúc 1.024.

In [ ]:
if not hasattr(F, "_sdpa_original"):
    F._sdpa_original = F.scaled_dot_product_attention
_sdpa = F._sdpa_original
ATTN_PATH = set()

def sdpa_low_mem(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, **kwargs):
    try:
        with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            out = _sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, **kwargs)
        ATTN_PATH.add("memory-efficient")
        return out
    except RuntimeError:
        ATTN_PATH.add("chia khúc")
    out = torch.empty_like(query)
    step = 1024
    for i in range(0, query.shape[-2], step):
        m = attn_mask[..., i:i + step, :] if attn_mask is not None else None
        out[..., i:i + step, :] = _sdpa(query[..., i:i + step, :], key, value, attn_mask=m, dropout_p=dropout_p, **kwargs)
    return out

F.scaled_dot_product_attention = sdpa_low_mem

## 4. Tải model

Chỉ tải đúng phần CatVTON dùng, có thanh tiến trình. Lần đầu mất vài phút, các lần sau đọc từ cache.
Cảnh báo `does not appear to have a file named diffusion_pytorch_model.safetensors` là **bình thường**: repo chỉ có file `.bin`.

In [ ]:
snapshot_download(
    repo_id="booksforcharlie/stable-diffusion-inpainting",  # bản sao, repo gốc của runwayml đã bị xoá
    allow_patterns=["model_index.json", "scheduler/*", "unet/*"],
    ignore_patterns=["*.ckpt"],
)
snapshot_download(repo_id="stabilityai/sd-vae-ft-mse", allow_patterns=["config.json", "*.safetensors"])
CKPT = snapshot_download(repo_id="zhengchong/CatVTON")
print("Đã tải xong")

## 5. Nạp model lên GPU

- **Mỗi GPU một pipeline**: UNet và VAE ở fp16, khoảng 2 GB mỗi GPU.
- **Một AutoMasker** (DensePose + SCHP) đặt ở GPU cuối.

Nạp lần lượt từng cái để RAM không vọt lên.

In [ ]:
import model.pipeline as catvton_pipeline
from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

# Tắt thanh tqdm bên trong pipeline: hai GPU chạy song song sẽ in chồng lên nhau
import tqdm as _tqdm
catvton_pipeline.tqdm = SimpleNamespace(tqdm=lambda *a, **k: _tqdm.tqdm(*a, **{**k, "disable": True}))

PIPES = []
for g in GPUS:
    PIPES.append(CatVTONPipeline(
        base_ckpt="booksforcharlie/stable-diffusion-inpainting",
        attn_ckpt=CKPT,
        attn_ckpt_version="mix",
        weight_dtype=DTYPE,
        use_tf32=False,             # T4 không có TF32
        skip_safety_check=True,
        device=g,
    ))
    gc.collect()

MASK_GPU = GPUS[-1]
automasker = AutoMasker(
    densepose_ckpt=os.path.join(CKPT, "DensePose"),
    schp_ckpt=os.path.join(CKPT, "SCHP"),
    device=MASK_GPU,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)

def gpu_report():
    for g in GPUS:
        free, total = torch.cuda.mem_get_info(g)
        print(f"  {g}: dùng {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB")
print(f"{len(PIPES)} pipeline, masker ở {MASK_GPU}")
gpu_report()

## 6. Hàm thử đồ

Hai bước tách riêng:
1. **`get_mask`**: chạy tuần tự, có cache. Một người mặc áo `upper` chỉ cần tính mask một lần, dùng lại cho mọi sản phẩm.
2. **`run_jobs`**: chia việc diffusion cho các GPU. Mỗi GPU xử lý một ảnh, xong lấy ảnh kế tiếp.

In [ ]:
OUT = Path("/kaggle/working/out")
OUT.mkdir(exist_ok=True)
_mask_cache = {}

def load_person(path):
    return resize_and_crop(Image.open(path).convert("RGB"), (W, H))

def load_garment(path):
    return resize_and_padding(Image.open(path).convert("RGB"), (W, H))

def get_mask(person_path, cloth_type):
    # cloth_type: 'upper' | 'lower' | 'overall' | 'inner' | 'outer'
    key = (str(person_path), cloth_type, W, H)
    if key not in _mask_cache:
        with torch.cuda.device(MASK_GPU):
            mask = automasker(load_person(person_path), cloth_type)["mask"]
        _mask_cache[key] = mask_processor.blur(mask, blur_factor=9)
    return _mask_cache[key]

def _generate(pipe, job):
    with torch.cuda.device(pipe.device), torch.inference_mode():
        generator = torch.Generator(device=pipe.device).manual_seed(job.get("seed", SEED))
        torch.cuda.synchronize(pipe.device); t0 = time.time()
        result = pipe(
            image=load_person(job["person"]),
            condition_image=load_garment(job["garment"]),
            mask=job["mask"],
            num_inference_steps=job.get("steps", STEPS),
            guidance_scale=job.get("guidance", GUIDANCE),
            height=H, width=W,
            generator=generator,
        )[0]
        torch.cuda.synchronize(pipe.device)
    return result, time.time() - t0

def run_jobs(jobs):
    # Mỗi job: {"person", "garment", "cloth_type", tuỳ chọn "steps", "guidance", "seed"}.
    # Bước 1, tuần tự: tạo mask (DensePose không an toàn khi chạy song song)
    for job in jobs:
        job["mask"] = get_mask(job["person"], job["cloth_type"])
    # Bước 2, song song: mỗi GPU lấy một pipeline rảnh từ hàng đợi
    free = queue.Queue()
    for p in PIPES:
        free.put(p)
    def work(job):
        pipe = free.get()
        try:
            job["result"], job["seconds"] = _generate(pipe, job)
            job["gpu"] = str(pipe.device)
        finally:
            free.put(pipe)
        return job
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(PIPES)) as pool:
        done = list(pool.map(work, jobs))
    print(f"{len(done)} ảnh trong {time.time() - t0:.0f}s · attention: {', '.join(sorted(ATTN_PATH)) or '?'}")
    return done

def strip(images, height=384):
    # Ghép các ảnh thành một hàng ngang để so sánh
    images = [im.resize((int(im.width * height / im.height), height)) for im in images]
    out = Image.new("RGB", (sum(im.width for im in images), height), "white")
    x = 0
    for im in images:
        out.paste(im, (x, 0)); x += im.width
    return out

## 7. Chạy thử bằng ảnh mẫu của repo
Hai ảnh chạy cùng lúc trên hai GPU. Nên thấy dòng `attention: memory-efficient`.

In [ ]:
men = sorted((REPO / "resource/demo/example/person/men").glob("*.png"))[:2]
upper = sorted((REPO / "resource/demo/example/condition/upper").glob("*.jpg"))[0]

done = run_jobs([{"person": p, "garment": upper, "cloth_type": "upper"} for p in men])
for j in done:
    print(f"{j['person'].name}: {j['seconds']:.1f}s trên {j['gpu']}")
gpu_report()
display(strip([load_garment(upper)] + [vis_mask(load_person(j["person"]), j["mask"]) for j in done] + [j["result"] for j in done]))

## 8. Ảnh của MAEVEN

Cấu trúc dataset (tên dataset tuỳ ý):
```
garments/   mv-01.jpg  mv-02.jpg  mv-03.jpg  mv-04.jpg   ← ảnh sản phẩm, nền trơn
persons/    *.jpg / *.png                                ← người mẫu đứng thẳng, chính diện
```
`GARMENTS` cho model biết mỗi món thuộc vùng nào. Áo khoác (`mv-03`) thử cả `outer` lẫn `upper`.

In [ ]:
GARMENTS = {
    "mv-01": "upper",  # Áo sơ mi lanh mộc
    "mv-02": "lower",  # Quần ống suông
    "mv-03": "outer",  # Áo khoác không cổ
    "mv-04": "upper",  # Áo thun cổ tròn
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def find_dataset(root=Path("/kaggle/input")):
    # Tìm thư mục chứa garments/ ở mọi độ sâu; nếu Kaggle giữ nguyên file zip thì tự giải nén
    hits = sorted(p.parent for p in root.rglob("garments") if p.is_dir())
    if hits:
        return hits[0]
    for z in root.rglob("*.zip"):
        out = Path("/kaggle/working/data") / z.stem
        with zipfile.ZipFile(z) as f:
            f.extractall(out)
        hits = sorted(p.parent for p in out.rglob("garments") if p.is_dir())
        if hits:
            return hits[0]
    return None

DATA = find_dataset()
garments, persons = {}, []
if DATA is None:
    print("Không thấy thư mục garments/. Những gì đang có trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").rglob("*"))[:40]:
        print("  ", p)
elif not (DATA / "persons").is_dir():
    print(f"{DATA} thiếu thư mục persons/")
else:
    garments = {p.stem: p for p in sorted((DATA / "garments").iterdir()) if p.suffix.lower() in IMG_EXT}
    persons = [p for p in sorted((DATA / "persons").iterdir()) if p.suffix.lower() in IMG_EXT]
    print(f"{len(garments)} sản phẩm, {len(persons)} người mẫu từ {DATA}")
    missing = set(garments) - set(GARMENTS)
    if missing:
        print("Chưa khai báo vùng cho (sẽ bỏ qua):", sorted(missing))

In [ ]:
jobs = [
    {"sku": sku, "person": p, "garment": garments[sku], "cloth_type": GARMENTS[sku]}
    for sku in garments if sku in GARMENTS
    for p in persons
]
log = []
if jobs:
    done = run_jobs(jobs)
    for j in done:
        name = f"{j['sku']}__{j['person'].stem}.jpg"
        j["result"].save(OUT / name, quality=92)
        log.append({"sku": j["sku"], "person": j["person"].stem, "type": j["cloth_type"],
                    "size": f"{W}x{H}", "steps": STEPS, "seconds": round(j["seconds"], 1), "gpu": j["gpu"], "file": name})
    for sku in dict.fromkeys(j["sku"] for j in done):
        row = [j["result"] for j in done if j["sku"] == sku]
        display(strip([load_garment(garments[sku])] + row))
    (OUT / "log.json").write_text(json.dumps(log, ensure_ascii=False, indent=2))
    print(f"Trung bình {np.mean([r['seconds'] for r in log]):.1f}s/ảnh mỗi GPU → {OUT}")
    gpu_report()

## 9. Thí nghiệm: steps và guidance

- **`steps`**: số bước khử nhiễu. Nhiều bước thì chi tiết hơn nhưng chậm hơn, gần như tuyến tính.
- **`guidance`**: mức bám ảnh áo. Thấp thì mềm nhưng dễ lệch họa tiết. Cao thì bám sát nhưng dễ gắt.

9 ảnh chạy song song trên 2 GPU.

In [ ]:
if garments and persons:
    sku = next(s for s in garments if s in GARMENTS)
    grid = [
        {"person": persons[0], "garment": garments[sku], "cloth_type": GARMENTS[sku], "steps": s, "guidance": g}
        for s in (10, 20, 30) for g in (1.5, 2.5, 3.5)
    ]
    done = run_jobs(grid)
    for s in (10, 20, 30):
        row = [j for j in done if j["steps"] == s]
        print(f"steps={s}  guidance 1.5 / 2.5 / 3.5  ·  " + " / ".join(f"{j['seconds']:.1f}s" for j in row))
        display(strip([j["result"] for j in row]))

## 10. Chấm kết quả

Ghi vào `research/README.md` (mục *Nhật ký thí nghiệm*) cho từng sản phẩm:

| Tiêu chí | Câu hỏi |
|---|---|
| Màu | Có lệch so với áo thật không? Khách đã phàn nàn "màu ngoài đời nhạt hơn web". |
| Chất vải | Nếp nhăn tự nhiên của lanh còn không, hay bị làm phẳng như vải tổng hợp? |
| Phom | Ống suông còn suông không? Cổ bo của áo thun có bị biến dạng? |
| Danh tính | Khuôn mặt, tay và nền ảnh có giữ nguyên không? |
| Thời gian | Bao nhiêu giây mỗi ảnh trên T4, ở chế độ `FAST` và đầy đủ? |

Ảnh và `log.json` nằm ở *Output → `/kaggle/working/out`*.